# Fine-tuning Granite LLM (CPU Version)

This notebook fine-tunes IBM Granite without quantization for CPU environments.

In [ ]:
!pip install --upgrade pip
!pip install torch==2.1.0
!pip install transformers==4.36.0
!pip install datasets==2.16.0
!pip install peft==0.7.0
!pip install accelerate==0.25.0
!pip install trl==0.7.4
!pip install scipy

In [ ]:
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    pipeline
)
from peft import LoraConfig, PeftModel, get_peft_model
from trl import SFTTrainer
import json

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
MODEL_NAME = "ibm-granite/granite-3b-code-instruct"
OUTPUT_DIR = "../model/granite-security-finetuned"
DATASET_PATH = "../datasets/security_training_data.jsonl"

LEARNING_RATE = 2e-4
NUM_EPOCHS = 3
BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
MAX_SEQ_LENGTH = 1024

LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

print("Config loaded")

In [ ]:
dataset = load_dataset('json', data_files=DATASET_PATH, split='train')
print(f"Total examples: {len(dataset)}")

In [ ]:
def format_chat_template(example):
    messages = example['messages']
    formatted_text = f"""<|system|>
{messages[0]['content']}

<|user|>
{messages[1]['content']}

<|assistant|>
{messages[2]['content']}"""
    return {"text": formatted_text}

dataset = dataset.map(format_chat_template, remove_columns=dataset.column_names)
dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset['train']
eval_dataset = dataset['test']

print(f"Train: {len(train_dataset)}, Eval: {len(eval_dataset)}")

In [ ]:
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading model (CPU mode)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,
    device_map="cpu",
    trust_remote_code=True,
    low_cpu_mem_usage=True
)

print("✓ Model loaded on CPU")

In [ ]:
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=5,
    save_strategy="epoch",
    evaluation_strategy="epoch",
    push_to_hub=False,
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=lora_config,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    tokenizer=tokenizer,
    args=training_args,
    packing=False,
)

print("✓ Trainer ready")

In [ ]:
import time
print("Starting training (this will take a while on CPU)...\n")
start_time = time.time()

trainer.train()

training_time = time.time() - start_time
print(f"\nTraining completed in {training_time/60:.2f} minutes")

In [ ]:
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✓ Model saved to {OUTPUT_DIR}")

In [ ]:
print("Testing fine-tuned model...")

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,
    device_map="cpu",
    trust_remote_code=True
)

model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
model = model.merge_and_unload()

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.7,
    do_sample=True
)

print("✓ Model loaded")

In [ ]:
test_prompt = """<|system|>
You are a cybersecurity expert.

<|user|>
Analyze: Employee emp_999 had 5 after-hours accesses from external IP, downloaded sensitive files.

<|assistant|>
"""

result = pipe(test_prompt)
response = result[0]['generated_text'].split('<|assistant|>')[-1].strip()
print("RESPONSE:")
print(response)

In [ ]:
merged_model_path = OUTPUT_DIR + "-merged"
model.save_pretrained(merged_model_path)
tokenizer.save_pretrained(merged_model_path)
print(f"✓ Merged model: {merged_model_path}")
print("Ready for vLLM deployment!")